In [60]:
import numpy as np
import pandas as pd
from pathlib import Path
 
# BERTopic and its sub-components
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
 
# UMAP and HDBSCAN — BERTopic uses these internally but we configure them explicitly
# so results are reproducible
from umap import UMAP
from hdbscan import HDBSCAN

In [61]:
df = pd.read_json("../data/processed/final_songs.json")
llm_embeddings = np.load("../data/cache/llm_embedding.npy")
umap_embeddings = np.load("../data/cache/umap_embedding.npy")

In [76]:
umap_model = UMAP(
        n_components=5,       # reduce to 5D before clustering (BERTopic default)
        n_neighbors=15,
        min_dist=0.0,         # tighter clusters for topic discovery
        metric="cosine",
        random_state=42,
    )

hdbscan_model = HDBSCAN(
        min_cluster_size=15,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,  # required for soft-clustering / probability scores
    )

# Use unigrams and bigrams; filter very common/rare words
vectorizer_model = CountVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=2,             # word must appear in at least 2 songs
    max_df=0.85,          # ignore words that appear in >85% of songs (too common)
)

In [77]:
representation_model = KeyBERTInspired()

In [78]:
model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        top_n_words=10,           # keywords per topic
        verbose=True,
    )

In [79]:
docs = df['preprocessed_lyrics'].fillna("").tolist()

topic_ids, probabilities = model.fit_transform(docs)

2026-03-20 21:03:15,028 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11508.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 44/44 [00:05<00:00,  7.91it/s]
2026-03-20 21:03:21,808 - BERTopic - Embedding - Completed ✓
2026-03-20 21:03:21,809 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-20 21:03:24,822 - BERTopic - Dimensionality - Completed ✓
2026-03-20 21:03:24,823 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-20 21:03:24,856 - BERTopic - Cluster - Completed ✓
2026-03-20 21:03:24,857 - BERTopic - Representation - Fine-tuning topics using representation m

In [80]:
summary = model.get_document_info(docs)

In [81]:
summary[summary["Representative_document"]==True]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
131,You choke my throat with words of wonder You m...,1,1_ill sing_bad tattoos_love hard_roses im,"[ill sing, bad tattoos, love hard, roses im, m...",[God loves you God loves you God loves you God...,ill sing - bad tattoos - love hard - roses im ...,0.790171,True
159,We just wanna party all day We just wanna part...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,True
183,"You think you got it all right, all right But ...",-1,-1_wanna die_babe_theres nothin_im gonna,"[wanna die, babe, theres nothin, im gonna, fee...","[La, la, la, la, la, la, la La, la, la, la, la...",wanna die - babe - theres nothin - im gonna - ...,0.000000,True
310,"La-la-la-la, la-la-la-la, la La-la-la-la, la-l...",-1,-1_wanna die_babe_theres nothin_im gonna,"[wanna die, babe, theres nothin, im gonna, fee...","[La, la, la, la, la, la, la La, la, la, la, la...",wanna die - babe - theres nothin - im gonna - ...,0.000000,True
695,"(Three and four and) Ba-ba-da, ba-ba-da-ba-da ...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,True
1077,"""God is telling you and I there is death, for ...",1,1_ill sing_bad tattoos_love hard_roses im,"[ill sing, bad tattoos, love hard, roses im, m...",[God loves you God loves you God loves you God...,ill sing - bad tattoos - love hard - roses im ...,1.000000,True
1150,God loves you God loves you God loves you God ...,1,1_ill sing_bad tattoos_love hard_roses im,"[ill sing, bad tattoos, love hard, roses im, m...",[God loves you God loves you God loves you God...,ill sing - bad tattoos - love hard - roses im ...,1.000000,True
1304,How can it be? You and me Might be meant to be...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,0.949620,True
1339,"La, la, la, la, la, la, la La, la, la, la, la,...",-1,-1_wanna die_babe_theres nothin_im gonna,"[wanna die, babe, theres nothin, im gonna, fee...","[La, la, la, la, la, la, la La, la, la, la, la...",wanna die - babe - theres nothin - im gonna - ...,0.000000,True


In [82]:
summary

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,"Girl, it's so confusing sometimes to be a girl...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
1,I only threw this party for you Only threw thi...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,0.992577,False
2,I went my own way and I made it I'm your favou...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
3,Mm Oh I guess the apple don't fall far from th...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
4,I don't wanna share the space I don't wanna fo...,0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
...,...,...,...,...,...,...,...,...
1383,"Got a job, got a crib, got a mind of my own, h...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
1384,"I was a liar, I gave in to the fire I know I s...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
1385,"Yeah Hey old friends, let's look back on the c...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False
1386,"Mmm, I, mmm Share my life Take me for what I a...",0,0_gimme love_girl girl_thinkin bout_somethin,"[gimme love, girl girl, thinkin bout, somethin...","[(Three and four and) Ba-ba-da, ba-ba-da-ba-da...",gimme love - girl girl - thinkin bout - someth...,1.000000,False


In [83]:
model.get_topics()

{-1: [('wanna die', np.float32(0.31410488)),
  ('babe', np.float32(0.28763074)),
  ('theres nothin', np.float32(0.28684416)),
  ('im gonna', np.float32(0.27076048)),
  ('feel feel', np.float32(0.26819372)),
  ('wanna think', np.float32(0.26205447)),
  ('cause im', np.float32(0.2608119)),
  ('just goin', np.float32(0.24270667)),
  ('comin like', np.float32(0.24056767)),
  ('imagine imagine', np.float32(0.2370547))],
 0: [('gimme love', np.float32(0.37019834)),
  ('girl girl', np.float32(0.35453874)),
  ('thinkin bout', np.float32(0.2861126)),
  ('somethin', np.float32(0.27700633)),
  ('feelin', np.float32(0.26647022)),
  ('think im', np.float32(0.2590017)),
  ('make feel', np.float32(0.25640872)),
  ('im gonna', np.float32(0.24975297)),
  ('say im', np.float32(0.24539015)),
  ('babe', np.float32(0.24122122))],
 1: [('ill sing', np.float32(0.32966325)),
  ('bad tattoos', np.float32(0.32747978)),
  ('love hard', np.float32(0.32190344)),
  ('roses im', np.float32(0.31740698)),
  ('making f